In [1]:
import pandas as pd

from unittest.mock import patch

import model
from cv import cv_importance, grid_search, make_folds, pick_simple, selection_curve, summarize
from selection import select_features
from metrics import binary_log_loss, fbeta

df = pd.read_csv("data/train.csv")
X = df.drop(columns="skip")
y = df["skip"]
folds = make_folds(X, y)
ALL_FEATURES = list(X.columns)

In [2]:
scenarios = {
    "careful": lambda y_true, y_pred: fbeta(y_true, y_pred, beta=0.5),
    "rescue": lambda y_true, y_pred: fbeta(y_true, y_pred, beta=2.0),
}

In [3]:
grid = {
    "max_depth": [3, 4, 5, 6, 8, 10, None],
    "min_samples_leaf": [5, 10, 20, 40, 80],
    "smoothing": [0, 1, 3, 10],
}
with patch.object(model, "add_features", lambda X: X):
    baseline = grid_search(X, y, folds, scenarios, ALL_FEATURES, grid)
summarize(baseline, scenarios)

,max_depth,min_samples_leaf,smoothing,logloss,careful_thr,careful_score,careful_std,rescue_thr,rescue_score,rescue_std
careful,10.0,20.0,0.0,0.702268,0.75,0.851891,0.020433,0.13,0.778204,0.022477
rescue,NaN,20.0,0.0,0.744177,0.75,0.850484,0.020621,0.11,0.781016,0.019838
probability,NaN,20.0,3.0,0.397351,0.67,0.850000,0.024992,0.18,0.779466,0.016343


In [4]:
with_features = grid_search(X, y, folds, scenarios, ALL_FEATURES, grid)
summarize(with_features, scenarios)

,max_depth,min_samples_leaf,smoothing,logloss,careful_thr,careful_score,careful_std,rescue_thr,rescue_score,rescue_std
careful,4.0,20.0,0.0,0.271127,0.54,0.907694,0.012965,0.32,0.862109,0.025903
rescue,6.0,20.0,0.0,0.306742,0.56,0.906994,0.014088,0.31,0.885588,0.020120
probability,6.0,20.0,3.0,0.259719,0.55,0.906994,0.014088,0.34,0.885588,0.020120


In [5]:
best = summarize(with_features, scenarios)
params = {case: {"max_depth": None if pd.isna(best.loc[case, "max_depth"]) else int(best.loc[case, "max_depth"]),
                 "min_samples_leaf": int(best.loc[case, "min_samples_leaf"]),
                 "smoothing": float(best.loc[case, "smoothing"])}
          for case in best.index}
params["careful"]["threshold"] = float(best.loc["careful", "careful_thr"])
params["rescue"]["threshold"] = float(best.loc["rescue", "rescue_thr"])
params

{'careful': {'max_depth': 4,
  'min_samples_leaf': 20,
  'smoothing': 0.0,
  'threshold': 0.54},
 'rescue': {'max_depth': 6,
  'min_samples_leaf': 20,
  'smoothing': 0.0,
  'threshold': 0.31},
 'probability': {'max_depth': 6, 'min_samples_leaf': 20, 'smoothing': 3.0}}

In [6]:
pi_scores = {
    "careful": (lambda m, X, y: fbeta(y, m.predict(X), beta=0.5), True),
    "rescue": (lambda m, X, y: fbeta(y, m.predict(X), beta=2.0), True),
    "probability": (lambda m, X, y: binary_log_loss(y, m.predict_proba(X)), False),
}
importances = {case: cv_importance(X, y, folds, ALL_FEATURES, params[case], fn, gib)
               for case, (fn, gib) in pi_scores.items()}
pd.DataFrame(importances).round(4)

,careful,rescue,probability
track_genre,0.3846,0.1541,0.6212
listening_context,0.2942,0.2908,0.6340
energy,0.1756,0.1706,0.3543
playlist_genre_1_share,0.1519,0.0496,0.1647
playlist_genre_3_share,0.1410,0.0452,0.1622
track_plays_7d,0.1346,0.0991,0.2212
playlist_genre_0_share,0.1323,0.0383,0.1444
playlist_genre_2_share,0.1259,0.0373,0.1407
playlist_repeat_share,0.1256,0.0942,0.2080
instrumentalness,0.0576,0.0695,0.1247


In [7]:
ks = [3, 5, 7, 10, 13, 16, 20, 25, 30, 40]
curves = {case: selection_curve(X, y, folds, scenarios, importances[case], params[case], ks)
          for case in importances}
pd.DataFrame({
    "careful": curves["careful"].set_index("k")["careful_score"],
    "careful_std": curves["careful"].set_index("k")["careful_std"],
    "rescue": curves["rescue"].set_index("k")["rescue_score"],
    "rescue_std": curves["rescue"].set_index("k")["rescue_std"],
    "logloss": curves["probability"].set_index("k")["logloss"],
}).round(4)

,careful,careful_std,rescue,rescue_std,logloss
k,,,,,
3,0.7532,0.0116,0.7307,0.0008,0.4909
5,0.7452,0.0139,0.7442,0.0053,0.4119
7,0.7375,0.0239,0.7466,0.0049,0.3924
10,0.9064,0.0141,0.8845,0.0196,0.2612
13,0.9074,0.0130,0.8860,0.0204,0.2586
16,0.9077,0.0131,0.8860,0.0204,0.2578
20,0.9077,0.0131,0.8854,0.0200,0.2568
25,0.9077,0.0131,0.8860,0.0204,0.2568
30,0.9077,0.0131,0.8860,0.0204,0.2561


In [8]:
SELECTED = select_features(importances["careful"], 10)
assert all(set(SELECTED) == set(select_features(importances[c], 10)) for c in importances)
repeated = make_folds(X, y, seeds=(1, 2, 3))
grid_fine = {
    "max_depth": [3, 4, 5, 6, 7, 8],
    "min_samples_leaf": [5, 10, 15, 20, 30, 40, 60],
    "smoothing": [0, 0.5, 1, 2, 3, 5],
}
tuned = grid_search(X, y, repeated, scenarios, SELECTED, grid_fine)
summarize(tuned, scenarios)

,max_depth,min_samples_leaf,smoothing,logloss,careful_thr,careful_score,careful_std,rescue_thr,rescue_score,rescue_std
careful,6.0,30.0,0.5,0.269204,0.63,0.908466,0.013084,0.26,0.880408,0.017735
rescue,7.0,15.0,0.0,0.388590,0.66,0.903467,0.013356,0.27,0.885887,0.015898
probability,6.0,30.0,3.0,0.262195,0.62,0.908114,0.013036,0.29,0.880408,0.017735


In [9]:
choice = pd.DataFrame({case: pick_simple(tuned, case, n_folds=15)
                       for case in ["careful", "rescue", "probability"]}).T
choice

,max_depth,min_samples_leaf,smoothing,logloss,careful_thr,careful_score,careful_std,rescue_thr,rescue_score,rescue_std,depth
careful,4.0,30.0,0.0,0.277558,0.44,0.905744,0.013213,0.18,0.863263,0.016981,4.0
rescue,6.0,20.0,0.0,0.332309,0.70,0.904386,0.010905,0.30,0.882755,0.017864,6.0
probability,6.0,30.0,2.0,0.262593,0.62,0.908466,0.013084,0.28,0.880408,0.017735,6.0
